# Whisper 크기별 비교

**알고 싶은 것.** ① 모델을 키우면 오류율과 처리 시간이 얼마나 달라지는가 ② WER에는 잘못 들은 것 말고 표기 차이가 얼마나 섞이는가

**조건.** 연구와 같은 `openai-whisper`, `language="ko"`, 나머지 기본값. 데이터는 공개 낭독 음성 Zeroth-Korean 시험 분할입니다.

**말할 수 없는 것.** 낭독 음성 결과이므로 상담 음성에서 같은 폭으로 좋아진다고 말하면 안 됩니다.

**시작.** Select Kernel → Colab → New Colab Server에서 GPU를 고르고, 셀을 읽으면서 위에서부터 실행합니다. 끝나면 `Cmd+S`로 저장하고 `Colab: Remove Server`로 서버를 지웁니다.

In [ ]:
# 1. 설치. 버전을 고정해 다시 돌려도 같은 코드가 돌게 한다
%pip install -q openai-whisper==20250625 soundfile pyarrow

In [ ]:
# 2. 데이터 받기. Whisper가 바라는 16kHz 한 채널인지 여기서 확인한다
import io
import urllib.request
from pathlib import Path

import pyarrow.parquet as pq
import soundfile as sf

DATA_URL = 'https://huggingface.co/api/datasets/Bingsu/zeroth-korean/parquet/default/test/0.parquet'
data_file = Path('zeroth_test.parquet')
if not data_file.exists():
    urllib.request.urlretrieve(DATA_URL, data_file)

samples = []
for row in pq.read_table(data_file).to_pylist():
    audio, rate = sf.read(io.BytesIO(row['audio']['bytes']), dtype='float32')
    if rate != 16000 or audio.ndim != 1:
        raise ValueError(f'예상과 다른 음성입니다: {rate}Hz, {audio.ndim}차원')
    samples.append({'text': row['text'], 'audio': audio, 'seconds': len(audio) / rate})

print(f"{len(samples)}문장, 음성 {sum(s['seconds'] for s in samples) / 60:.1f}분")
print('정답 예:', samples[0]['text'])

In [ ]:
# 3. 평가 코드 받기. 연구 저장소의 함수를 고치지 않고 입력 글자만 바꿔서 잰다
import re
import sys

EVAL_COMMIT = 'c474b908851acb227baffaf9ea2e98867d67e7d8'
urllib.request.urlretrieve(
    f'https://raw.githubusercontent.com/wpalswpa/stt-llm-depression-screening/{EVAL_COMMIT}/evaluation.py',
    'evaluation.py',
)
sys.path.insert(0, '.')
from evaluation import evaluate_rows, score_transcript

def without_punctuation(text):
    return ' '.join(re.sub(r'[^\w\s]', '', text).split())

def without_spaces(text):
    return ''.join(without_punctuation(text).split())

# 실행 전에 답을 먼저 생각해 보세요: 아래 세 줄의 WER은 각각 얼마일까요?
print('쉼표·마침표만 다름   ', score_transcript('안녕 세상', '안녕, 세상.'))
print('구두점을 지운 뒤     ', score_transcript('안녕 세상', without_punctuation('안녕, 세상.')))
print('숫자 표기만 다름     ', score_transcript('이천 십 팔 년', '2018년'))

In [ ]:
# 4. 실험 조건. 시간이 부족하면 MODELS를 줄이고, 먼저 시험하려면 LIMIT에 20처럼 작은 수를 넣는다
MODELS = ['tiny', 'small', 'large-v3']
LIMIT = None

In [ ]:
# 5. 전사. 모델마다 결과를 파일로 남겨, 끊겨도 다시 실행하면 끝난 모델은 건너뛴다
# 실패한 문장은 버리지 않고 빈 전사로 남긴다(평가에서 전부 삭제로 계산된다)
import gc
import json
import time

import torch
import whisper

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('장치:', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU (처리 시간을 T4와 비교하지 마세요)')

chosen = samples[:LIMIT]
Path('results').mkdir(exist_ok=True)

def result_path(name):
    return Path('results') / f'{name}_{len(chosen)}.json'

for name in MODELS:
    if result_path(name).exists():
        print(name, '이미 있음, 건너뜀')
        continue
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    model = whisper.load_model(name, device=device)
    load_seconds = time.perf_counter() - started

    rows = []
    for sample in chosen:
        started = time.perf_counter()
        try:
            pred_text, error = model.transcribe(sample['audio'], language='ko', fp16=device == 'cuda')['text'], ''
        except Exception as exc:
            pred_text, error = '', repr(exc)
        rows.append({'gt_text': sample['text'], 'pred_text': pred_text, 'error': error,
                     'audio_seconds': sample['seconds'], 'seconds': time.perf_counter() - started})
        if len(rows) % 100 == 0:
            print(f'  {name}: {len(rows)}/{len(chosen)}')

    peak_gb = torch.cuda.max_memory_allocated() / 1024**3 if device == 'cuda' else None
    result_path(name).write_text(json.dumps({'load_seconds': load_seconds, 'peak_gb': peak_gb, 'rows': rows}, ensure_ascii=False))
    print(name, '완료')

    del model
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# 6. 요약표. 첫 문장은 준비 시간이 섞일 수 있어 처리 속도 계산에서 뺀다
import pandas as pd

def applied(rows, change):
    return [{'gt_text': change(r['gt_text']), 'pred_text': change(r['pred_text'])} for r in rows]

results, summary = {}, []
for name in MODELS:
    results[name] = json.loads(result_path(name).read_text())
    rows = results[name]['rows']
    timed = rows[1:] or rows
    summary.append({
        '모델': name,
        'WER 그대로': evaluate_rows(rows)['WER'],
        'WER 구두점 제거': evaluate_rows(applied(rows, without_punctuation))['WER'],
        'CER 그대로': evaluate_rows(rows)['CER'],
        'CER 구두점·공백 제거': evaluate_rows(applied(rows, without_spaces))['CER'],
        '빈 전사': evaluate_rows(rows)['empty_hypothesis'],
        '오류': sum(1 for r in rows if r['error']),
        '숫자 나온 문장': sum(1 for r in rows if re.search(r'\d', r['pred_text'])),
        '실시간 대비 배속': sum(r['audio_seconds'] for r in timed) / sum(r['seconds'] for r in timed),
        '불러오기(초)': results[name]['load_seconds'],
        '최대 GPU 메모리(GB)': results[name]['peak_gb'],
    })

table = pd.DataFrame(summary).set_index('모델').round(3)
table.to_csv('results/summary.csv')
print(f'문장 {len(chosen)}개')
print(table.T.to_string())

In [ ]:
# 7. 눈으로 보기. 수치만 보지 말고 같은 문장을 모델별로 읽어 어떤 오류인지 확인한다
for index in range(0, len(chosen), max(1, len(chosen) // 5)):
    print('정답    :', chosen[index]['text'])
    for name in MODELS:
        print(f'{name:8}:', results[name]['rows'][index]['pred_text'].strip())
    print()

## 결과(2026-11-02, 맥 CPU)

Colab이 아니라 맥 CPU에서 AI 도구로 실행했습니다. 처리 속도는 T4 수치가 아닙니다.

| 모델 | WER 그대로 | WER 구두점 제거 | CER 구두점·공백 제거 | 실시간 대비 처리 속도 |
|---|---|---|---|---|
| tiny | 0.592 | 0.551 | 0.256 | 48.1배 |
| small | 0.389 | 0.323 | 0.103 | 10.3배 |
| large-v3 | 0.316 | 0.265 | 0.095 | 0.86배 |